# 2025-04-14 generate Gaussian grids

Adapted from this notebook from Spencer: https://github.com/ai2cm/explore/blob/master/spencer/2023-08-30-generate-gaussian-grids.ipynb.

In [5]:
!pip install "git+https://github.com/ai2cm/fv3net.git@68e4f378e933a9d90af9268a4f330cd6fc308e7d#egg=xtorch_harmonics&subdirectory=external/xtorch_harmonics"

  Cloning https://github.com/ai2cm/fv3net.git (to revision 68e4f378e933a9d90af9268a4f330cd6fc308e7d) to /tmp/pip-install-2kwxuom_/xtorch-harmonics_d8788e90a82d4e439f97828301352bd9
  Running command git clone --filter=blob:none --quiet https://github.com/ai2cm/fv3net.git /tmp/pip-install-2kwxuom_/xtorch-harmonics_d8788e90a82d4e439f97828301352bd9
  Running command git rev-parse -q --verify 'sha^68e4f378e933a9d90af9268a4f330cd6fc308e7d'
  Running command git fetch -q https://github.com/ai2cm/fv3net.git 68e4f378e933a9d90af9268a4f330cd6fc308e7d
  Running command git checkout -q 68e4f378e933a9d90af9268a4f330cd6fc308e7d
  Resolved https://github.com/ai2cm/fv3net.git to commit 68e4f378e933a9d90af9268a4f330cd6fc308e7d
  Running command git submodule update --init --recursive -q
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/NVIDIA/torch-harmonics.git (to revision 8826246cacf6c37b600cdd63fde210815ba238fd) to /tmp/pip-install-2kwxuom_/torch-harmonics_82f66d468b544ecab89fc15

In [6]:
import numpy as np
import xarray as xr
import xtorch_harmonics

In [11]:
def midpoints(bounds):
    return (bounds[:-1] + bounds[1:]) / 2


def compute_coordinates(n_lat, n_lon):
    lat_midpoints = xtorch_harmonics.xtorch_harmonics.compute_quadrature_latitudes(
        n_lat, "legendre-gauss"
    )
    lat_edges = np.concatenate(
        (np.array((-90,)), midpoints(lat_midpoints), np.array((90,)))
    )

    lon_edges = np.linspace(0, 360, n_lon + 1)
    lon_midpoints = midpoints(lon_edges)
    return lat_midpoints, lat_edges, lon_midpoints, lon_edges


def generate_grid(n_lat, n_lon):
    lat_midpoints, lat_edges, lon_midpoints, lon_edges = compute_coordinates(
        n_lat, n_lon
    )

    grid_lat = xr.DataArray(lat_edges, dims=["grid_y"], name="grid_lat")
    grid_latt = xr.DataArray(lat_midpoints, dims=["grid_yt"], name="grid_latt")
    grid_lon = xr.DataArray(lon_edges, dims=["grid_x"], name="grid_lon")
    grid_lont = xr.DataArray(lon_midpoints, dims=["grid_xt"], name="grid_lont")

    # Broadcast the coordinates against themselves to make a mesh grid
    # for the bounds and cell center coordinates.
    grid_lat, grid_lon = xr.broadcast(grid_lat, grid_lon)
    grid_latt, grid_lont = xr.broadcast(grid_latt, grid_lont)

    grid_lat.attrs["long_name"] = "Latitude cell edge"
    grid_lat.attrs["units"] = "degrees"
    grid_latt.attrs["long_name"] = "Latitude cell midpoint"
    grid_latt.attrs["units"] = "degrees"
    grid_lon.attrs["long_name"] = "Longitude cell edge"
    grid_lon.attrs["units"] = "degrees"
    grid_lont.attrs["long_name"] = "Longitude cell midpoint"
    grid_lont.attrs["units"] = "degrees"
    ds = xr.merge([grid_lat, grid_latt, grid_lon, grid_lont])
    return ds

In [12]:
four_degree = generate_grid(45, 90)
two_degree = generate_grid(90, 180)
quarter_degree = generate_grid(720, 1440)

In [13]:
quarter_degree

<xarray.Dataset> Size: 33MB
Dimensions:    (grid_y: 721, grid_x: 1441, grid_yt: 720, grid_xt: 1440)
Dimensions without coordinates: grid_y, grid_x, grid_yt, grid_xt
Data variables:
    grid_lat   (grid_y, grid_x) float64 8MB -90.0 -90.0 -90.0 ... 90.0 90.0 90.0
    grid_latt  (grid_yt, grid_xt) float64 8MB -89.81 -89.81 ... 89.81 89.81
    grid_lon   (grid_y, grid_x) float64 8MB 0.0 0.25 0.5 ... 359.5 359.8 360.0
    grid_lont  (grid_yt, grid_xt) float64 8MB 0.125 0.375 0.625 ... 359.6 359.9
Attributes:
    long_name:  Latitude cell edge
    units:      degrees

In [12]:
four_degree.to_netcdf("gaussian_grid_45_by_90.nc")
two_degree.to_netcdf("gaussian_grid_90_by_180.nc")
quarter_degree.to_netcdf("gaussian_grid_720_by_1440.nc")